# Remote Parser playground
### Use this notebook to debug remote content parsing.

In [1]:
%reload_ext autoreload
%autoreload 2

In [3]:
from spyglass.shijiegu.Analysis_SGU import get_linearization_map

from spyglass.shijiegu.Analysis_SGU import TrialChoice, DecodeIngredients, DecodeResults2D, ChangeofMind, ChangeofMindRemoteTheta, DecodeResultsLinear, ChangeofMindTriggeredDecode
from spyglass.shijiegu.decodeHelpers import runSessionNames
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename

[2026-01-18 12:28:21,956][INFO]: DataJoint 0.14.4 connected to shijiegu-alt@lmf-db.cin.ucsf.edu:3306


In [4]:
key = {
    "nwb_file_name": "lewis20240108_.nwb",
    "epoch": 6,"proportion":0.1, "parameter":"params_both_max_run_time_2_state"}
query = ChangeofMindTriggeredDecode() & key
decode_df = query.fetch1_dataframe(key)

[2026-01-18 12:28:36,510][WARNING]: Skipped checksum for file with hash: 59652e9c-a04d-1e4f-b074-d0c0f1a3f1e6, and path: /stelmo/nwb/analysis/lewis20240108/lewis20240108_KPM8OV6PB4.nwb


In [ ]:
decode_df_trial43 = decode_df.loc[10]
decode_df_trial43

In [ ]:
plt.plot(decode_df_trial43.triggered_positions)
plt.plot(decode_df_trial43.triggered_decodes_abs)

In [ ]:
nwb_copy_file_name = "lewis20240108_.nwb"
session_name = "06_Rev2Session3"
entry = DecodeIngredients & {'nwb_file_name':nwb_copy_file_name,
                             'interval_list_name':session_name}
# position_1d,position_2d,
position_1d = pd.read_csv(entry.fetch1('position_1d')) #still need 1D position
position_2d = pd.read_csv(entry.fetch1('position_2d')) # need 2D position

In [ ]:
from spyglass.shijiegu.load import load_decode
# decode
classifier_param_name = "default_decoding_gpu_4armMaze"
encoding_set = "2Dheadspeed_above_4"

decode = load_decode(nwb_copy_file_name,
                        session_name,
                        classifier_param_name = classifier_param_name,
                        encoding_set = encoding_set,
                        use_1d = 1)

In [ ]:
triggered_position = pd.DataFrame(decode_df_trial43.triggered_positions, index = decode_df_trial43.time_abs)
triggered_trial_info = decode_df_trial43.triggered_trial_info

position_axis = np.array(decode.coords['position'])
        
# find the arm the animal is at
subset_arm = triggered_trial_info[-1] + 5
    
# find the trial
# find t0, t1 to consider
trialID = triggered_trial_info[-2]
(t0, t1) = (triggered_position.index[0],triggered_position.index[-1])
#
t0 = 1704758806.5332046 #- 0.5
t1 = 1704758806.5332046 + 0.03

In [ ]:
from spyglass.shijiegu.changeOfMind_triggered import region
from spyglass.shijiegu.ripple_add_replay import (find_start_end,
                                                 position_posterior2arm_posterior,
                                                 select_subset_helper)

position2d_subset = position_2d[np.logical_and(position_2d.time>=t0, position_2d.time<=t1)]
position1d_subset = position_1d[np.logical_and(position_1d.time>=t0, position_1d.time<=t1)]
decode_subset = select_subset_helper(decode,(t0,t1))

In [ ]:
subset_ind = position1d_subset.track_segment_id == subset_arm

In [ ]:
position2d_subset = position2d_subset[subset_ind]
position1d_subset = position1d_subset[subset_ind]
decode_subset = decode_subset.isel(time = np.argwhere(subset_ind).ravel())

In [ ]:
from spyglass.shijiegu.decodeQuality import return_low_hpd_time
posterior_position_subset = decode_subset.causal_posterior.sum(dim='state')
max_posterior_position = np.array(position_axis[posterior_position_subset.argmax(dim = 'position')])
is_concentrated = return_low_hpd_time(decode_subset, return_boolean = True) > 0

is_remote = np.zeros_like(max_posterior_position) #just to initialize

In [ ]:
# is posterior concentrated
is_concentrated = return_low_hpd_time(decode_subset, return_boolean = True)

In [ ]:
return_low_hpd_time(decode_subset, return_boolean = True, debug = True, prob_mass = 0.2)

In [ ]:
all_indices = return_low_hpd_time(decode_subset, return_boolean = True, debug = True, prob_mass = 0.2)

In [ ]:
fig, axe = plt.subplots(1,1)
decode_subset.causal_posterior.sum(dim='state').T.plot(ax = axe)

x_axis = np.array(decode_subset.position)
for t_ind in range(len(all_indices)):
    for index in range(len(all_indices[t_ind])):
        axe.scatter(decode_subset.time[t_ind],x_axis[all_indices[t_ind][index]],color = 'grey')

In [ ]:
from spyglass.shijiegu.changeOfMind_triggered import linear_map
is_remote = np.zeros_like(max_posterior_position) #just to initialize
for k in [8]:#region.keys():
    if k == int(subset_arm):
        continue
    (arm_base, arm_top) = region[k]
    is_remote = is_remote + np.logical_and(max_posterior_position <= arm_top, max_posterior_position >= arm_base)
    # find remote representation at home
    #is_remote = is_remote + np.logical_and(max_posterior_position >= 0, max_posterior_position <= linear_map[0][1])

In [ ]:
from ripple_detection.core import segment_boolean_series

minimum_duration = 0.02

# restrict to moving time
is_moving = np.array(position2d_subset.head_speed) > 4
min_len = np.min([len(is_moving),len(is_remote)])
# choose min because one variable is a subset of decode and the other is a subset of position.
# there could be 1 or 2 time point difference.
is_moving = is_moving[:min_len]
is_remote = is_remote[:min_len]
is_concentrated = is_concentrated[:min_len]
    
is_remote = np.logical_and(is_remote, is_moving)
#is_remote = np.logical_and(is_remote, is_concentrated)
    
is_remote_pd = pd.Series(is_remote, index = posterior_position_subset.time)
is_remote_segments = np.array(segment_boolean_series(
        is_remote_pd, minimum_duration=minimum_duration))